# Figures 2 and 3: reliability, accuracy vs. ECE, and design decisions

`figures/final_5_figure_a_b.png` (Figure 2, `fig:main_analysis_1`):
* (a) aggregated reliability diagram and (b) calibration error vs. accuracy: `results/main.csv`

`figures/final_5_figure_c_d_e.png` (Figure 3, `fig:main_analysis_2`):
* (a) calibration error vs. training decodes: `results/main.csv`, `results/one_stage.csv`, `results/basc_tp.csv`
* (b) Stage-1 AUROC vs. k: `results/stage1_scaling.csv`
* (c) Stage-2 recalibrators: `results/stage2_recalibrators.csv`

Calibration error is `MAIN_METRIC` (paper_utils.py; ECE1 in the paper).

In [ ]:
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as sstats

from paper_utils import (FIGURES_DIR, MAIN_METRIC, METHOD_COLOR, METHOD_LABEL, METRIC_TEX, RESULTS_DIR,
                         pooled_reliability, promote_semantic)

plt.style.use('seaborn-v0_8')
plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 11,
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})
INK  = plt.rcParams['axes.labelcolor']
GRID = plt.rcParams.get('grid.color', '#cccccc')

def style_axes(ax, xlabel=None, ylabel=None):
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    ax.grid(True, color=GRID, lw=0.7)
    ax.set_axisbelow(True)
    if xlabel is not None: ax.set_xlabel(xlabel, color=INK)
    if ylabel is not None: ax.set_ylabel(ylabel, color=INK)
    return ax

In [ ]:
RELIABILITY_METHODS = ['logprob', 'verbal_conf_stem6', 'basc_k2']       # Figure 2

FAMILIES = {                                                            # Figure 3 (a)
    'LR - SC':  ['ridge_clip_sc_k2', 'ridge_clip_sc_k5', 'ridge_clip_sc_k10', 'ridge_clip_sc_k20'],
    'LR - SSC': ['ssc_ridge_k2', 'ssc_ridge_k5', 'ssc_ridge_k10', 'ssc_ridge_k20'],
    'BASC-L':   ['basc_k2', 'basc_k5', 'basc_k10', 'basc_k20'],
}
FAMILY_COLOR = {
    'BASC-L':   METHOD_COLOR['basc_k2'],
    'LR - SC':  METHOD_COLOR['ridge_clip_sc_k10'],
    'LR - SSC': METHOD_COLOR['ssc_ridge_k10'],
}
ONE_STAGE_KS = [2, 5, 10, 20]
ONE_STAGE_COLOR = METHOD_COLOR['basc_k5']
# BASC-TP: 2 decodes per calibration question. (method, #questions)
BASC_TP_POINTS = [('tp_iso_diso', 200), ('tp_iso_dcal', 400)]

STAGE1_KS = [2, 3, 5, 10, 20, 50, 100]                                  # Figure 3 (b)
BINARY_COLOR, CONT_COLOR = '#C44E52', '#A6BB7E'

RECAL_SCORES = [('bascL_k5', 'BASC-L (k=5)'), ('tp', 'BASC-TP')]        # Figure 3 (c)
RECALS = [
    ('iso',         'Isotonic (ours)',   '#C44E52'),
    ('platt',       'Platt',             '#4C72B0'),
    ('platt_bin10', 'Scal.-bin. (B=10)', '#DD8452'),
    ('hist10',      'Hist. bin. (B=10)', '#7B5FC7'),
]
RECAL_METRIC, ALPHA = MAIN_METRIC, 0.05

# Split sizes of the 20/20/60 split: 200 D_score + 200 D_iso questions.
N_SCORE, N_ISO = 200, 200

In [ ]:
df = promote_semantic(pd.read_csv(RESULTS_DIR / 'main.csv'))
one_stage = pd.read_csv(RESULTS_DIR / 'one_stage.csv')
one_stage = one_stage[one_stage['k'].isin(ONE_STAGE_KS)].copy()
basc_tp = pd.read_csv(RESULTS_DIR / 'basc_tp.csv')
basc_tp = basc_tp[basc_tp['method'].isin([m for m, _ in BASC_TP_POINTS])].copy()
stage1 = pd.read_csv(RESULTS_DIR / 'stage1_scaling.csv')
stage1 = stage1[stage1['stage2'] == 'iso_binary']

In [ ]:
def combo_mean_sem(frame, method, metric):
    """Seed-average per (model, dataset), then mean / SEM across combinations."""
    per = frame[frame['method'] == method].groupby(['model', 'dataset'])[metric].mean()
    return float(per.mean()), float(per.std() / np.sqrt(max(len(per), 1)))

def _n_decode(method):
    """Full-response decodes on the calibration set: BASC-L uses k per D_score
    question and 2 per D_iso question; one-stage methods use k per question."""
    k = int(re.match(r'.*_k(\d+)$', method).group(1))
    if method.startswith('basc_'):
        return N_SCORE * k + N_ISO * 2
    return (N_SCORE + N_ISO) * k

def stage1_line(target, metric='AUROC'):
    per = stage1[stage1['target'] == target].groupby(['k', 'model', 'dataset'])[metric].mean().reset_index()
    return per.groupby('k')[metric].agg(['mean', 'count', 'std']).reindex(STAGE1_KS)

# Figure 3 (c): mean / SEM across combinations; paired Wilcoxon vs. isotonic.
_r = pd.read_csv(RESULTS_DIR / 'stage2_recalibrators.csv')
_per = _r.groupby(['score', 'recal', 'model', 'dataset'])[RECAL_METRIC].mean().unstack('recal')
_rows = []
for score, score_label in RECAL_SCORES:
    p = _per.loc[score]
    for recal, label, _ in RECALS:
        x = p[recal]
        row = {'score': score_label, 'recal': label, 'mean': x.mean(),
               'sem': x.std() / np.sqrt(len(x)), 'sig_worse': False}
        if recal != 'iso':
            row['p'] = sstats.wilcoxon(p['iso'], x).pvalue
            row['sig_worse'] = bool(row['p'] < ALPHA and x.mean() > p['iso'].mean())
        _rows.append(row)
recal_summary = pd.DataFrame(_rows)
recal_summary

In [ ]:
def panel_reliability(ax):
    ax.plot([0, 1], [0, 1], color='k', lw=1.0, ls='--', zorder=1, label=r"$x=y$")
    for m in RELIABILITY_METHODS:
        counts, mp, mc = pooled_reliability(df, m)
        ok = counts > 0
        ax.plot(mp[ok], mc[ok], color=METHOD_COLOR[m], lw=2.0,
                marker='*' if 'basc' in m else 'o', markersize=11 if 'basc' in m else 5,
                markeredgecolor='white', markeredgewidth=0.8,
                label=METHOD_LABEL[m].replace(" (k=2)", ""), zorder=3)
    style_axes(ax, xlabel='Predicted confidence', ylabel='Empirical accuracy')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_title('(a) Aggregated Reliability', color=INK, fontsize=13)
    ax.legend(loc='upper left', fontsize=10, frameon=False, labelcolor=INK, ncol=2)

def panel_acc_vs_ece(ax):
    per = (df.groupby(['method', 'model', 'dataset'])
             .agg(ECE=(MAIN_METRIC, 'mean'), accuracy=('accuracy', 'mean')).reset_index())
    for m in RELIABILITY_METHODS:
        sub = per[per['method'] == m]
        ax.scatter(sub['accuracy'], sub['ECE'], color=METHOD_COLOR[m],
                   marker='*' if 'basc' in m else 'o', s=100 if 'basc' in m else 35,
                   edgecolors='white', linewidths=0.7,
                   label=METHOD_LABEL[m].replace(" (k=2)", ""), zorder=3)
        x, y = sub['accuracy'].to_numpy(), sub['ECE'].to_numpy()
        slope, intercept = np.polyfit(x, y, 1)
        xs = np.linspace(x.min(), x.max(), 2)
        ax.plot(xs, slope * xs + intercept, color=METHOD_COLOR[m], lw=1.6, alpha=0.75, zorder=2)
    style_axes(ax, xlabel='Accuracy', ylabel=METRIC_TEX[MAIN_METRIC])
    ax.set_xlim(0.15, 1)
    ax.set_title(f'(b) Accuracy vs. {METRIC_TEX[MAIN_METRIC]}', color=INK, fontsize=13)
    ax.legend(loc='upper right', fontsize=10, frameon=False, labelcolor=INK, ncol=2)

def panel_efficiency(ax):
    y_top, y_bot = 0.0, 1.0

    def _family(fam, methods_fam):
        nonlocal y_top, y_bot
        xs = [_n_decode(m) for m in methods_fam]
        st = [combo_mean_sem(df, m, MAIN_METRIC) for m in methods_fam]
        ys, es = [s[0] for s in st], [s[1] for s in st]
        is_basc = fam == 'BASC-L'
        ax.errorbar(xs, ys, yerr=es, color=FAMILY_COLOR[fam],
                    marker='*' if is_basc else 'o', markersize=11 if is_basc else 5,
                    markeredgecolor='white', markeredgewidth=0.8,
                    lw=1.8, capsize=3, ecolor=FAMILY_COLOR[fam], elinewidth=1.0, label=fam, zorder=3)
        y_top = max(y_top, max(y + e for y, e in zip(ys, es)))
        y_bot = min(y_bot, min(y - e for y, e in zip(ys, es)))

    # Draw (and legend) order: baselines, 1-stage, BASC-TP, then BASC-L on top.
    for fam, methods_fam in FAMILIES.items():
        if fam != 'BASC-L':
            _family(fam, methods_fam)
    os_stats = [combo_mean_sem(one_stage, f'onestage_binomial_k{k}', MAIN_METRIC) for k in ONE_STAGE_KS]
    ax.errorbar([(N_SCORE + N_ISO) * k for k in ONE_STAGE_KS], [s[0] for s in os_stats],
                yerr=[s[1] for s in os_stats], color=ONE_STAGE_COLOR, marker='D', markersize=6,
                markeredgecolor='white', markeredgewidth=0.8, lw=1.8, capsize=3,
                ecolor=ONE_STAGE_COLOR, elinewidth=1.0, label='1-stage', zorder=3)
    y_top = max([y_top] + [y + e for y, e in os_stats])
    tp_stats = [combo_mean_sem(basc_tp, m, MAIN_METRIC) for m, _ in BASC_TP_POINTS]
    ax.errorbar([2 * n for _, n in BASC_TP_POINTS], [s[0] for s in tp_stats],
                yerr=[s[1] for s in tp_stats], color=METHOD_COLOR['basc_tp'], marker='s', markersize=7,
                markeredgecolor='white', markeredgewidth=0.8, lw=1.8, capsize=3,
                ecolor=METHOD_COLOR['basc_tp'], elinewidth=1.0, label=METHOD_LABEL['basc_tp'], zorder=3)
    y_top = max([y_top] + [y + e for y, e in tp_stats])
    y_bot = min([y_bot] + [y - e for y, e in tp_stats])
    _family('BASC-L', FAMILIES['BASC-L'])

    span = y_top - y_bot
    ax.set_ylim(y_bot - 0.05 * span, y_top + 0.18 * span)
    style_axes(ax, xlabel='Training decodes', ylabel=METRIC_TEX[MAIN_METRIC])
    ax.set_title('(a) Training Efficiency', color=INK, fontsize=13)
    ax.legend(loc='upper right', fontsize=10, frameon=False, labelcolor=INK, ncol=3)

def panel_stage1_auroc(ax):
    for target, label, color in [('ridge', 'Ridge-SC', CONT_COLOR), ('binomial', 'Binomial-logistic', BINARY_COLOR)]:
        st = stage1_line(target)
        x, y = st.index.to_numpy(), st['mean'].to_numpy()
        e = st['std'].to_numpy() / np.sqrt(np.clip(st['count'].to_numpy(), 1, None))
        ax.errorbar(x, y, yerr=e, marker='o', color=color, label=label, lw=2,
                    markersize=6, capsize=3, markeredgecolor='white', markeredgewidth=1)
        ax.fill_between(x, y - e, y + e, color=color, alpha=0.12)
    ax.set_xscale('log')
    ax.set_xticks(STAGE1_KS); ax.set_xticklabels([str(k) for k in STAGE1_KS])
    ax.minorticks_off()
    style_axes(ax, xlabel='k', ylabel='AUROC')
    ax.set_title('(b) Stage 1 Estimator', color=INK, fontsize=13)
    ax.legend(loc='lower right', fontsize=10, frameon=False, labelcolor=INK)

def panel_recalibrators(ax):
    n_rec, width = len(RECALS), 0.19
    group_x = np.arange(len(RECAL_SCORES))
    for j, (recal, label, color) in enumerate(RECALS):
        sub = recal_summary[recal_summary['recal'] == label]
        xs = group_x + (j - (n_rec - 1) / 2) * width
        ax.bar(xs, sub['mean'], width, yerr=sub['sem'], color=color, label=label, linewidth=1.2,
               error_kw={'capsize': 2, 'lw': 1, 'ecolor': INK, 'alpha': 0.6})
        for x, (_, s) in zip(xs, sub.iterrows()):
            if s['sig_worse']:
                ax.text(x, s['mean'] + s['sem'] + 0.001, '*', ha='center', va='bottom',
                        fontsize=14, fontweight='bold', color=INK)
    style_axes(ax, ylabel=METRIC_TEX[MAIN_METRIC])
    ax.grid(False, axis='x')
    ax.set_xticks(group_x)
    ax.set_xticklabels([lab for _, lab in RECAL_SCORES], color=INK)
    ax.set_xlabel(f'* worse than isotonic (paired Wilcoxon, p < {ALPHA})', color=INK, fontsize=9)
    ax.set_ylim(0, recal_summary.eval('mean + sem').max() * 1.45)
    ax.set_title('(c) Stage 2 Recalibrator', color=INK, fontsize=13)
    ax.legend(loc='upper center', fontsize=9.5, frameon=False, labelcolor=INK, ncol=2,
              handlelength=1.2, columnspacing=1.0)

## Figure 2: (a) reliability, (b) calibration error vs. accuracy

In [ ]:
fig, axd = plt.subplot_mosaic([['a', 'b']], width_ratios=[3, 3], figsize=(10, 2.25), layout='constrained')
panel_reliability(axd['a'])
panel_acc_vs_ece(axd['b'])
fig.get_layout_engine().set(w_pad=0.04, h_pad=0.06, wspace=0.02, hspace=0.02)
fig.savefig(FIGURES_DIR / 'final_5_figure_a_b.png')
plt.show()

## Figure 3: (a) training decodes, (b) Stage-1 estimator, (c) Stage-2 recalibrator

In [ ]:
fig, axd = plt.subplot_mosaic([['c', 'd', 'e']], width_ratios=[4, 3, 3], figsize=(13, 3), layout='constrained')
panel_efficiency(axd['c'])
if MAIN_METRIC == 'ECE1':
    axd['c'].set_ylim(None, 0.22)      # as in the paper
panel_stage1_auroc(axd['d'])
panel_recalibrators(axd['e'])
fig.get_layout_engine().set(w_pad=0.04, h_pad=0.06, wspace=0.02, hspace=0.02)
fig.savefig(FIGURES_DIR / 'final_5_figure_c_d_e.png')
plt.show()